# ServerWatch demo commands

Copy each cell into the right place. Every step is marked with where it runs:

- **💻 Local**: your computer (Git Bash on Windows), in the project folder.
- **☁️ EC2**: the server, after you have connected with SSH.
- **🌐 AWS**: the AWS console in the browser.

Replace `ELASTIC_IP` with the instance address. The talking points are in the
[presentation guide](presentation-guide.md).

---
# Part 1: Keys, instance and connection

## 1. 🌐 AWS: launch the instance and create the SSH key (G1, G2)
- **Launch instance**: Ubuntu 24.04 LTS, a free-tier type.
- **Key pair → Create new key pair**: name `serverwatch`, type ED25519, format `.pem`. AWS keeps the public key; the private key downloads to your computer.
- **Security group**: SSH 22 from **My IP** only (HTTP and HTTPS come later).
- **Elastic IPs → Allocate**, then **Associate** it with the instance.

## 2. 💻 Local: connect (G2)
`chmod 400` makes the private key readable only by you (SSH refuses it otherwise), then you log in as `ubuntu`. Answer `yes` to the fingerprint question.

In [ ]:
chmod 400 ~/Downloads/serverwatch.pem
ssh -i ~/Downloads/serverwatch.pem ubuntu@ELASTIC_IP

For G2, say: *"Each person gets their own key pair. To add a teammate, I would paste their public key into `authorized_keys` on their own account; their private key never leaves their computer."*

## 3. 💻 Local: copy the setup script with SCP (G3)
Sends `setup.sh` to the server's home folder over SSH. Run in a second local terminal, in the project folder.

In [ ]:
scp -i ~/Downloads/serverwatch.pem setup.sh ubuntu@ELASTIC_IP:~/setup.sh

---
# Part 2: Set up the server

## 4. ☁️ EC2: clone the repository (G4)
Installs Git and clones the project into the `ubuntu` user's home, so that user owns the code and no service needs root.

In [ ]:
sudo apt-get update && sudo apt-get install -y git
git clone https://github.com/EmirU116/server-api.git ~/serverwatch
cd ~/serverwatch && git log -1 --oneline

## 5. ☁️ EC2: save the Discord webhook (G6)
Stores the webhook in a file only you can read; the prompt hides what you paste.

In [ ]:
umask 077
install -d -m 700 "$HOME/.config/serverwatch-alerts"
if IFS= read -rsp 'Discord webhook URL: ' webhook; then
    printf '%s\n' "$webhook" > "$HOME/.config/serverwatch-alerts/discord-webhook"
fi
unset webhook
printf '\n'

## 6. ☁️ EC2: run setup (G3)
Installs everything: PostgreSQL, the API and collector services, Nginx, Cron backups, Docker, the `senior` and `junior` accounts, the Discord timer and the `sw-*` aliases.

In [ ]:
export SERVERWATCH_SENIOR_USER=senior
export SERVERWATCH_JUNIOR_USER=junior
export SERVERWATCH_DISCORD_WEBHOOK_FILE="$HOME/.config/serverwatch-alerts/discord-webhook"
SERVERWATCH_PROJECT_DIR="$HOME/serverwatch" bash ~/setup.sh ELASTIC_IP

Reloads the shell so the new aliases work, then lists them.

In [ ]:
source ~/.bashrc
sw-help

---
# Part 3: Show the G requirements

## 7. ☁️ EC2: verify the deployment (G7, G9)
Read-only checks of the services, database, collector and Nginx; expect 8 passed, 0 failed.

In [ ]:
cd ~/serverwatch
bash verify.sh ELASTIC_IP

## 8. ☁️ EC2: Nginx by hand (G8)
Opens the reverse-proxy config in nano, checks the syntax, and reloads without dropping connections.

In [ ]:
cd /etc/nginx/sites-enabled
ls -l
sudo nano serverwatch.conf
sudo nginx -t
sudo systemctl reload nginx
cd ~/serverwatch

🌐 Add **HTTP 80 from anywhere** to the security group, then open `http://ELASTIC_IP/` and `http://ELASTIC_IP/docs` in the browser.

## 9. ☁️ EC2: systemd and the database (G7, G9)
Shows the API and collector running under systemd, and that the API and PostgreSQL only listen on 127.0.0.1.

In [ ]:
sw-status
systemctl cat serverwatch-api
sudo ss -ltnp | grep -E ':5432|:8000'

## 10. ☁️ EC2: Cron backup (G5)
Shows the daily 03:15 schedule. In `sudoedit`, set it to the next minute; put `15 3` back after it has run.

In [ ]:
sudo cat /etc/cron.d/serverwatch-backup
sudoedit /etc/cron.d/serverwatch-backup

Shows that Cron ran the backup and lists the snapshots.

In [ ]:
sudo journalctl -t serverwatch-backup -n 10 --no-pager
sudo ls -lt /var/backups/serverwatch

Restores the newest snapshot into a separate database and counts the rows: a backup only counts if it can be restored.

In [ ]:
(
    set -euo pipefail
    unset PGHOST PGHOSTADDR PGPORT PGSERVICE PGSERVICEFILE
    snapshot=/var/backups/serverwatch/$(sudo ls -1 /var/backups/serverwatch | tail -n 1)
    restore_db="serverwatch_restore_$(date +%s)"
    sudo -u postgres createdb --host=/var/run/postgresql --port=5432 "$restore_db"
    sudo cat "$snapshot/database.dump" |
        sudo -u postgres pg_restore --host=/var/run/postgresql --port=5432 \
            --exit-on-error --no-owner --no-privileges --dbname="$restore_db"
    sudo -u postgres psql -X --host=/var/run/postgresql --port=5432 \
        --set=ON_ERROR_STOP=1 --dbname="$restore_db" \
        -c 'SELECT count(*) AS metrics, max(collected_at) AS latest FROM metrics' \
        -c 'SELECT count(*) AS incidents FROM incidents'
    printf 'Restored database: %s\n' "$restore_db"
)

## 11. ☁️ EC2: Discord alert (G6)
Lowers the disk threshold below current usage, so the timer posts to Discord within a minute.

In [ ]:
cd ~/serverwatch
sw-timers
df -h /
bash infrastructure/bootstrap/setup_discord_alerts.sh "$HOME/.config/serverwatch-alerts/discord-webhook" 25

After the message arrives, put the normal 90% threshold back.

In [ ]:
bash infrastructure/bootstrap/setup_discord_alerts.sh "$HOME/.config/serverwatch-alerts/discord-webhook" 90

---
# Part 4: Show the VG requirements

## 12. ☁️ EC2: give `ubuntu` access to Docker (VG1)
Adds `ubuntu` to the `docker` group so it can run Compose without sudo. The change applies at the next login, so reconnect (step 2) and check.

In [ ]:
sudo usermod -aG docker "$USER"
exit

In [ ]:
docker info --format '{{.ServerVersion}}'

## 13. ☁️ EC2: Docker Compose (VG1)
Builds the images and starts db → migrate → api → web; `migrate` showing `Exited (0)` is success.

In [ ]:
cd ~/serverwatch
bash scripts/deployment/compose_host.sh start
bash scripts/deployment/compose_host.sh ps -a

Sends this host's metrics to the Compose API; stop with Ctrl+C after about 20 seconds.

In [ ]:
bash scripts/deployment/compose_host.sh collect

## 14. ☁️ EC2: data survives container recreation (VG1)
Counts the rows, removes the containers (the volume stays), starts them again and counts again; both counts must match.

In [ ]:
bash scripts/deployment/compose_host.sh exec db psql -U postgres -d serverwatch -c "SELECT count(*) FROM metrics;"
bash scripts/deployment/compose_host.sh down
bash scripts/deployment/compose_host.sh up -d --wait
bash scripts/deployment/compose_host.sh exec db psql -U postgres -d serverwatch -c "SELECT count(*) FROM metrics;"

## 15. ☁️ EC2: domain and HTTPS (VG2)
🌐 First, in Namecheap → **Advanced DNS**, point the `@` and `www` A records at the Elastic IP, and in AWS add **HTTPS 443 from anywhere**. This checks that both names already point here.

In [ ]:
getent hosts serverwatch.dev www.serverwatch.dev

Gets a Let's Encrypt certificate with Certbot, turns on HTTPS and checks renewal. For rehearsals, put `SERVERWATCH_CERTBOT_STAGING=1` in front to save real certificates.

In [ ]:
cd ~/serverwatch
bash infrastructure/bootstrap/setup_https.sh serverwatch.dev ELASTIC_IP

Shows the certificate, the renewal timer and the HTTPS Nginx site. Then open `https://serverwatch.dev` in the browser.

In [ ]:
sudo certbot certificates
systemctl list-timers certbot.timer
sudo nano /etc/nginx/sites-enabled/serverwatch-domain.conf

## 16. Optional: deploy after CI (VG4 bonus)
Needs the setup in [continuous deployment](continuous-deployment.md). 💻 **Local:** push a small change, then watch CI and *ServerWatch deploy* under **Actions** on GitHub.

In [ ]:
git commit -am "Demo change"
git push

☁️ **EC2:** shows which commit GitHub deployed and the images tagged with it.

In [ ]:
cat ~/.config/serverwatch-delivery/deployed-commit
bash ~/serverwatch/scripts/deployment/compose_host.sh ps

---
## 17. 🌐 Afterwards
Watch the recording and check the link. Then in AWS: **stop or terminate the instance** and **release the Elastic IP** (an unattached Elastic IP costs money).